# Automated Medical Image Analysis for Early Disease Detection
## Phase 3: Cloud GPU Training Pipeline for YOLOv8 (Kaggle & Colab)

This notebook is specifically configured for **Kaggle Notebooks with NVIDIA Tesla T4 ×2 GPU** (or Google Colab T4 GPU).

### Pipeline Structure & Execution Order:
1. **Environment Setup & GPU Check**: Verify CUDA availability, device VRAM, install dependencies, and initialize centralized runtime paths (`compute_device`, `YOLO_DATA_DIR`, `yaml_path`, `EXPERIMENT_DIR`).
2. **GPU & Pipeline Smoke Test**: Rapid 1-epoch / 4-sample synthetic trial in an isolated temporary sandbox to verify CUDA forward/backward passes and AMP before starting data conversion.
3. **Dynamic Dataset Auto-Detection**: Robust discovery of mounted CSV and DICOM files under `/kaggle/input` or local paths.
4. **Patient-Level Stratified Splitting**: 80% Train, 10% Val, 10% Test with **zero patient data leakage**.
5. **Parallel DICOM to YOLO Conversion**: VOI LUT, `MONOCHROME1` inversion, 640×640 JPEG rendering, empty labels for negative normal radiographs.
6. **Dataset Integrity Validation & YAML Configuration**: Generates `rsna_yolo.yaml` and stops execution with helpful errors if any image-label mismatch or out-of-bounds box is detected.
7. **YOLOv8 Training Run**: Pretrained `yolov8s.pt` with class `pneumonia_opacity`, 25 epochs, per-epoch checkpoints, and auto-resume.
8. **Object Detection Evaluation**: Precision, Recall, `mAP@0.50`, and `mAP@0.50:0.95` on validation and test sets.
9. **Separate Image-Level Classification Evaluation**: Clinical Sensitivity (Recall), Specificity, Accuracy, and Confusion Matrix.
10. **Visual Inspection & Plots**: Loss curves, confusion matrix, and prediction overlays.
11. **Weights Export**: Locate and download `best.pt` for local Apple M1 deployment.

### 1. Environment Setup & GPU Accelerator Verification
Installs imaging and deep learning libraries, verifies CUDA hardware acceleration, and initializes centralized paths.

In [ ]:
# Install medical imaging and object detection dependencies
!pip install -q ultralytics pydicom opencv-python-headless scikit-learn matplotlib pandas numpy pyyaml tqdm

import os
import sys
import json
import yaml
import shutil
import time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import torch
import pydicom
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO

# Handle pydicom VOI LUT modern import vs legacy fallback
try:
    from pydicom.pixels import apply_voi_lut
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_voi_lut

print("=" * 75)
print("ENVIRONMENT & HARDWARE CHECK")
print("=" * 75)
print(f"Python Version     : {sys.version.split()[0]}")
print(f"PyTorch Version    : {torch.__version__}")
print(f"CUDA Available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU         : {device_name} (Total: {device_count} device(s))")
    print(f"GPU VRAM (Device 0): {vram_gb:.2f} GB")
    compute_device = 0
else:
    print("[WARNING] CUDA is NOT available. Running on CPU!")
    print("To enable GPU on Kaggle: Settings (right sidebar) -> Accelerator -> GPU T4 x2.")
    compute_device = "cpu"

# Centralized Runtime Paths and Configuration
IS_KAGGLE = Path("/kaggle/working").exists()
BASE_WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./outputs")
YOLO_DATA_DIR = BASE_WORKING_DIR / "rsna_yolo"
yaml_path = YOLO_DATA_DIR / "rsna_yolo.yaml"
EXPERIMENT_DIR = BASE_WORKING_DIR / "rsna_pneumonia"
RUN_NAME = "yolov8s_pneumonia_run"

print(f"Compute Device     : {compute_device}")
print(f"YOLO Data Dir      : {YOLO_DATA_DIR}")
print(f"Dataset YAML Path  : {yaml_path}")
print(f"Experiment Dir     : {EXPERIMENT_DIR}")
print("=" * 75)

### 2. GPU & YOLO Pipeline Smoke Test (Self-Contained Verification)
Executes a rapid 1-epoch / 4-sample synthetic trial in an isolated temporary sandbox.
This verifies CUDA forward/backward passes, Automatic Mixed Precision (AMP), optimizer step, and checkpoint generation **before** starting any lengthy dataset conversion or full training.

In [ ]:
# Self-contained pre-flight smoke test
# Does not depend on RSNA dataset conversion or later cells.
import os
import shutil
import time
from pathlib import Path
import numpy as np
import cv2
import yaml
import torch
from ultralytics import YOLO

# Resolve compute device defensively if cell is executed independently
smoke_device = globals().get("compute_device", 0 if torch.cuda.is_available() else "cpu")
print(f"Running pre-training smoke test on device: {smoke_device}...")

# Create isolated temporary sandbox
smoke_dir = Path("/kaggle/working/smoke_test") if Path("/kaggle/working").exists() else (Path.cwd() / "smoke_test_tmp")
if smoke_dir.exists():
    shutil.rmtree(smoke_dir, ignore_errors=True)

smoke_img_dir = smoke_dir / "images"
smoke_lbl_dir = smoke_dir / "labels"
smoke_img_dir.mkdir(parents=True, exist_ok=True)
smoke_lbl_dir.mkdir(parents=True, exist_ok=True)

# Generate 4 synthetic medical-like images and YOLO labels
for idx in range(4):
    synthetic_img = np.random.randint(40, 200, (320, 320, 3), dtype=np.uint8)
    cv2.circle(synthetic_img, (160, 160), 50, (220, 220, 220), -1)
    img_path = smoke_img_dir / f"smoke_sample_{idx}.jpg"
    cv2.imwrite(str(img_path), synthetic_img, [cv2.IMWRITE_JPEG_QUALITY, 90])
    
    lbl_path = smoke_lbl_dir / f"smoke_sample_{idx}.txt"
    lbl_path.write_text("0 0.500000 0.500000 0.350000 0.350000\n")

# Write self-contained YAML configuration
smoke_yaml_cfg = {
    "path": str(smoke_dir.resolve()),
    "train": "images",
    "val": "images",
    "nc": 1,
    "names": {0: "pneumonia_opacity"}
}
smoke_yaml_file = smoke_dir / "smoke.yaml"
with open(smoke_yaml_file, "w") as f:
    yaml.dump(smoke_yaml_cfg, f, default_flow_style=False)

start_t = time.time()
try:
    smoke_model = YOLO("yolov8n.pt")
    smoke_results = smoke_model.train(
        data=str(smoke_yaml_file),
        epochs=1,
        imgsz=320,
        batch=2,
        device=smoke_device,
        project=str(smoke_dir),
        name="smoke_run",
        verbose=False,
        exist_ok=True
    )
    elapsed = time.time() - start_t
    print(f"\n[PASS] Smoke test completed successfully in {elapsed:.1f}s!")
    print("CUDA GPU forward/backward pass, AMP, and checkpoint saving verified.")
    print("Ready to proceed to dataset preparation and full training.")
except Exception as e:
    print(f"\n[FAIL] Smoke test failed: {e}")
    raise RuntimeError(f"GPU / YOLO smoke test failed: {e}")
finally:
    # Clean up smoke test run folder
    shutil.rmtree(smoke_dir, ignore_errors=True)

### 3. Robust Dataset Auto-Detection
Dynamically discovers mounted RSNA competition files under `/kaggle/input` without assuming fixed nested paths.

In [ ]:
def auto_detect_rsna_paths():
    search_roots = [
        Path("/kaggle/input"),
        Path("./data"),
        Path("../data"),
        Path.cwd() / "data",
        Path.cwd()
    ]
    
    found_csv = None
    found_dcm_dir = None
    
    print("Searching for RSNA dataset files across mounted paths...")
    for root in search_roots:
        if not root.exists():
            continue
        # 1. Search for labels CSV
        for csv_candidate in root.rglob("*.csv"):
            if "train" in csv_candidate.name.lower() or "label" in csv_candidate.name.lower():
                try:
                    df_head = pd.read_csv(csv_candidate, nrows=3)
                    if "patientId" in df_head.columns and "Target" in df_head.columns:
                        found_csv = csv_candidate.resolve()
                        break
                except Exception:
                    continue
        if found_csv is not None:
            break
            
    if found_csv is None:
        print("\n[ERROR] Could not locate RSNA stage_2_train_labels.csv!")
        if Path("/kaggle/input").exists():
            print("Current contents of /kaggle/input:")
            for p in Path("/kaggle/input").glob("*"):
                print(f" - {p.name}")
        raise FileNotFoundError("Please attach the 'RSNA Pneumonia Detection Challenge' competition dataset in Kaggle!")
        
    # 2. Search for DICOM image directory
    df_check = pd.read_csv(found_csv)
    sample_pids = set(df_check["patientId"].dropna().head(10).astype(str))
    
    for root in search_roots:
        if not root.exists():
            continue
        # Check named folders first
        for dir_name in ["stage_2_train_images", "train_images"]:
            for d_cand in root.rglob(dir_name):
                if d_cand.is_dir() and any(d_cand.glob("*.dcm")):
                    found_dcm_dir = d_cand.resolve()
                    break
            if found_dcm_dir:
                break
        if found_dcm_dir:
            break
            
        # Fallback search for any dir containing matching .dcm
        for dcm_f in root.rglob("*.dcm"):
            parent_dir = dcm_f.parent
            if any((parent_dir / f"{pid}.dcm").exists() for pid in sample_pids):
                found_dcm_dir = parent_dir.resolve()
                break
        if found_dcm_dir:
            break
            
    if found_dcm_dir is None:
        raise FileNotFoundError(f"Found CSV at {found_csv}, but could not locate matching .dcm images directory!")
        
    return found_csv, found_dcm_dir

labels_csv_path, images_dicom_dir = auto_detect_rsna_paths()
print("[SUCCESS] RSNA dataset paths located and verified:")
print(f" - Labels CSV Path : {labels_csv_path}")
print(f" - DICOM Images Dir: {images_dicom_dir}")

df_labels = pd.read_csv(labels_csv_path)
print(f" - Total Rows      : {len(df_labels):,}")
print(f" - Unique Patients : {df_labels['patientId'].nunique():,}")
display(df_labels.head())

### 4. Patient-Level Stratified Splitting (Zero Patient Leakage)
Partitions patients (not rows) into Train (80%), Val (10%), and Test (10%) splits, stratifying by positive vs negative diagnosis.

In [ ]:
def split_patients_stratified(df, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10, seed=42):
    patient_targets = df.groupby("patientId")["Target"].max()
    rng = np.random.RandomState(seed)
    split_map = {}
    
    for target_val in [0, 1]:
        pts = patient_targets[patient_targets == target_val].index.to_numpy()
        rng.shuffle(pts)
        n = len(pts)
        n_tr = int(round(n * train_ratio))
        n_va = int(round(n * val_ratio))
        
        for pid in pts[:n_tr]:
            split_map[pid] = "train"
        for pid in pts[n_tr:n_tr + n_va]:
            split_map[pid] = "val"
        for pid in pts[n_tr + n_va:]:
            split_map[pid] = "test"
            
    return split_map

patient_splits = split_patients_stratified(df_labels, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10)

# Leakage Assertions
tr_pts = {p for p, s in patient_splits.items() if s == 'train'}
va_pts = {p for p, s in patient_splits.items() if s == 'val'}
te_pts = {p for p, s in patient_splits.items() if s == 'test'}

assert len(tr_pts.intersection(va_pts)) == 0, "LEAKAGE ERROR: Overlap between Train and Val!"
assert len(tr_pts.intersection(te_pts)) == 0, "LEAKAGE ERROR: Overlap between Train and Test!"
assert len(va_pts.intersection(te_pts)) == 0, "LEAKAGE ERROR: Overlap between Val and Test!"

print("[PASS] Patient-level stratified split completed with ZERO data leakage:")
print(f" - Train Partition: {len(tr_pts):,} patients ({len(tr_pts)/len(patient_splits):.1%})")
print(f" - Val Partition  : {len(va_pts):,} patients ({len(va_pts)/len(patient_splits):.1%})")
print(f" - Test Partition : {len(te_pts):,} patients ({len(te_pts)/len(patient_splits):.1%})")

### 5. Parallel DICOM to YOLO Image & Label Generation
- Decodes DICOM 16-bit pixel arrays with VOI LUT windowing and `MONOCHROME1` inversion.
- Saves 640×640 JPEGs to `/kaggle/working/rsna_yolo/images/<split>/`.
- Generates normalized bounding box `.txt` files in `labels/<split>/`.
- **Negative Normal Cases (`Target=0`)**: Saves an **empty `.txt` file** so YOLO learns negative background contexts.

In [ ]:
# Ensure output directories exist under YOLO_DATA_DIR
for s in ["train", "val", "test"]:
    (YOLO_DATA_DIR / "images" / s).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)

def process_patient_record(args):
    patient_id, split, bboxes = args
    dcm_file = images_dicom_dir / f"{patient_id}.dcm"
    out_img = YOLO_DATA_DIR / "images" / split / f"{patient_id}.jpg"
    out_lbl = YOLO_DATA_DIR / "labels" / split / f"{patient_id}.txt"
    
    # Skip if both already exist (fast resume)
    if out_img.exists() and out_lbl.exists():
        return True
        
    if not dcm_file.exists():
        # Do not create orphaned labels if image is missing
        return False
        
    try:
        dcm = pydicom.dcmread(str(dcm_file))
        try:
            arr = apply_voi_lut(dcm.pixel_array, dcm)
        except Exception:
            arr = dcm.pixel_array
            
        if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = np.amax(arr) - arr
            
        arr = arr.astype(float)
        mn, mx = np.min(arr), np.max(arr)
        norm = ((arr - mn) / (mx - mn) * 255.0).astype(np.uint8) if mx > mn else np.zeros(arr.shape, dtype=np.uint8)
        resized = cv2.resize(norm, (640, 640), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(out_img), resized, [cv2.IMWRITE_JPEG_QUALITY, 90])
    except Exception as e:
        # If decoding fails, skip writing both
        return False
        
    # Convert bounding boxes
    lbl_lines = []
    for (x, y, w, h) in bboxes:
        xc = max(0.0, min(1.0, (x + w / 2.0) / 1024.0))
        yc = max(0.0, min(1.0, (y + h / 2.0) / 1024.0))
        wn = max(0.0, min(1.0, w / 1024.0))
        hn = max(0.0, min(1.0, h / 1024.0))
        lbl_lines.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
        
    # Write empty file if negative, or lines if positive
    out_lbl.write_text("\n".join(lbl_lines) + ("\n" if lbl_lines else ""))
    return True

# Build tasks
grouped = df_labels.groupby("patientId")
tasks = []
for pid, s in patient_splits.items():
    boxes = []
    if pid in grouped.groups:
        for _, r in grouped.get_group(pid).iterrows():
            if r["Target"] == 1 and pd.notna(r["x"]):
                boxes.append((float(r["x"]), float(r["y"]), float(r["width"]), float(r["height"])))
    tasks.append((pid, s, boxes))

print(f"Prepared {len(tasks):,} patients. Starting conversion...")
workers = min(os.cpu_count() or 4, 8)
with ThreadPoolExecutor(max_workers=workers) as executor:
    results = list(tqdm(executor.map(process_patient_record, tasks), total=len(tasks), desc="Converting RSNA"))

successful = sum(1 for r in results if r)
print(f"[SUCCESS] Converted {successful:,} / {len(tasks):,} patient radiographs to YOLO format!")

### 6. Generate `rsna_yolo.yaml` and Run Dataset Integrity Validation
Verifies 100% 1-to-1 matching, box bounds, empty labels, and stops execution if anything is invalid.

In [ ]:
yaml_config = {
    "path": str(YOLO_DATA_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "pneumonia_opacity"}
}

yaml_path = YOLO_DATA_DIR / "rsna_yolo.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_config, f, default_flow_style=False, sort_keys=False)

print(f"Generated Dataset YAML: {yaml_path}")

# Validation Checklist
validation_errors = []
split_stats = {}

for split in ["train", "val", "test"]:
    img_files = list((YOLO_DATA_DIR / "images" / split).glob("*.jpg"))
    lbl_files = list((YOLO_DATA_DIR / "labels" / split).glob("*.txt"))
    
    img_stems = {f.stem for f in img_files}
    lbl_stems = {f.stem for f in lbl_files}
    
    if img_stems != lbl_stems:
        diff = (img_stems ^ lbl_stems)
        validation_errors.append(f"{split} split has {len(diff)} unpaired image/label files!")
        
    pos_count = 0
    neg_count = 0
    box_count = 0
    
    for lf in lbl_files:
        content = lf.read_text().strip()
        if not content:
            neg_count += 1
        else:
            pos_count += 1
            for line in content.splitlines():
                tokens = line.strip().split()
                if len(tokens) != 5:
                    validation_errors.append(f"{lf.name}: Malformed line: {line}")
                else:
                    cls_id, xc, yc, w, h = int(tokens[0]), float(tokens[1]), float(tokens[2]), float(tokens[3]), float(tokens[4])
                    if cls_id != 0 or not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                        validation_errors.append(f"{lf.name}: Out of bounds box: {line}")
                    box_count += 1
                    
    split_stats[split] = {"total": len(img_files), "pos": pos_count, "neg": neg_count, "boxes": box_count}

if validation_errors:
    print("\n[CRITICAL ERROR] Dataset Validation Failed:")
    for err in validation_errors[:10]:
        print(f" - {err}")
    raise RuntimeError(f"Dataset validation failed with {len(validation_errors)} error(s)!")

print("\n" + "=" * 75)
print("[PASS] DATASET INTEGRITY VALIDATED (100% Paired & Compliant)")
print("=" * 75)
for s, st in split_stats.items():
    print(f"{s.capitalize():<6} -> Total: {st['total']:,} | Positives: {st['pos']:,} | Negatives: {st['neg']:,} | Boxes: {st['boxes']:,}")
print("=" * 75)

### 7. Full YOLOv8 Training Run with Persistent Checkpointing
- Architecture: Pretrained `yolov8s.pt` (Small, 11.2M parameters).
- Single class: `pneumonia_opacity`.
- Checkpointing: `save=True`, `save_period=1` (weights saved at every epoch).
- Auto-Resume: Automatically resumes if `last.pt` is detected.

In [ ]:
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
last_checkpoint = EXPERIMENT_DIR / RUN_NAME / "weights" / "last.pt"

if last_checkpoint.exists():
    print(f"[RESUME] Found existing checkpoint: {last_checkpoint}")
    print("Resuming training from interrupted state...")
    model = YOLO(str(last_checkpoint))
    train_results = model.train(resume=True)
else:
    print("[START] Initializing fresh YOLOv8s fine-tuning on RSNA dataset...")
    model = YOLO("yolov8s.pt")
    train_results = model.train(
        data=str(yaml_path),
        epochs=25,
        imgsz=640,
        batch=16,
        device=compute_device,
        project=str(EXPERIMENT_DIR),
        name=RUN_NAME,
        save=True,
        save_period=1,
        patience=10,
        amp=True,
        workers=4,
        exist_ok=True,
        verbose=True
    )

print("\n[SUCCESS] Training loop completed!")

### 8. Comprehensive Object Detection Localization Evaluation
Evaluates bounding box detection (`Precision`, `Recall`, `mAP@0.50`, `mAP@0.50:0.95`) on both Validation and Holdout Test splits.

In [ ]:
best_weights = EXPERIMENT_DIR / RUN_NAME / "weights" / "best.pt"
if not best_weights.exists():
    fallback_weights = EXPERIMENT_DIR / RUN_NAME / "weights" / "last.pt"
    if fallback_weights.exists():
        print(f"[NOTICE] best.pt not found, falling back to last.pt: {fallback_weights}")
        best_weights = fallback_weights
    else:
        raise FileNotFoundError(f"Trained checkpoint not found in {EXPERIMENT_DIR / RUN_NAME / 'weights'}. Please run training first!")

eval_model = YOLO(str(best_weights))

print("=" * 75)
print("OBJECT DETECTION METRICS (VALIDATION SPLIT)")
print("=" * 75)
val_metrics = eval_model.val(data=str(yaml_path), split="val", imgsz=640, device=compute_device)
p_val = float(val_metrics.box.p[0]) if len(val_metrics.box.p) > 0 else float(val_metrics.box.mp)
r_val = float(val_metrics.box.r[0]) if len(val_metrics.box.r) > 0 else float(val_metrics.box.mr)
map50_val = float(val_metrics.box.map50)
map95_val = float(val_metrics.box.map)
print(f"Precision (P)      : {p_val:.4f}")
print(f"Recall (R)         : {r_val:.4f}")
print(f"mAP @ 0.50         : {map50_val:.4f}")
print(f"mAP @ 0.50:0.95    : {map95_val:.4f}")

print("\n" + "=" * 75)
print("OBJECT DETECTION METRICS (HOLDOUT TEST SPLIT)")
print("=" * 75)
test_metrics = eval_model.val(data=str(yaml_path), split="test", imgsz=640, device=compute_device)
p_test = float(test_metrics.box.p[0]) if len(test_metrics.box.p) > 0 else float(test_metrics.box.mp)
r_test = float(test_metrics.box.r[0]) if len(test_metrics.box.r) > 0 else float(test_metrics.box.mr)
map50_test = float(test_metrics.box.map50)
map95_test = float(test_metrics.box.map)
print(f"Precision (P)      : {p_test:.4f}")
print(f"Recall (R)         : {r_test:.4f}")
print(f"mAP @ 0.50         : {map50_test:.4f}")
print(f"mAP @ 0.50:0.95    : {map95_test:.4f}")

### 9. Separate Image-Level Clinical Classification Evaluation
Separates bounding box metrics from patient-level diagnosis: computes Sensitivity, Specificity, Accuracy, and Confusion Matrix.

In [ ]:
test_images = list((YOLO_DATA_DIR / "images" / "test").glob("*.jpg"))
test_labels = (YOLO_DATA_DIR / "labels" / "test")

y_true = []
y_pred = []

print(f"Computing image-level clinical diagnosis on {len(test_images):,} test radiographs...")
for img_p in tqdm(test_images, desc="Evaluating Classification"):
    lbl_p = test_labels / f"{img_p.stem}.txt"
    has_gt = 1 if (lbl_p.exists() and len(lbl_p.read_text().strip()) > 0) else 0
    
    preds = eval_model.predict(source=str(img_p), conf=0.25, verbose=False, device=compute_device)
    has_pred = 1 if (preds and len(preds[0].boxes) > 0) else 0
    
    y_true.append(has_gt)
    y_pred.append(has_pred)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

tp = int(np.sum((y_true == 1) & (y_pred == 1)))
fp = int(np.sum((y_true == 0) & (y_pred == 1)))
tn = int(np.sum((y_true == 0) & (y_pred == 0)))
fn = int(np.sum((y_true == 1) & (y_pred == 0)))

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0.0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0

print("=" * 75)
print("CLINICAL IMAGE-LEVEL CLASSIFICATION METRICS")
print("=" * 75)
print(f"Accuracy            : {accuracy:.2%}")
print(f"Sensitivity (Recall): {sensitivity:.2%} (True Positive Rate)")
print(f"Specificity         : {specificity:.2%} (True Negative Rate)")
print(f"PPV (Precision)     : {ppv:.2%}")
print(f"NPV                 : {npv:.2%}")
print(f"Confusion Matrix    : TP={tp}, FP={fp}, TN={tn}, FN={fn}")
print("=" * 75)

### 10. Visual Diagnostic Plots & Summary Export

In [ ]:
run_folder = EXPERIMENT_DIR / RUN_NAME

# 1. Plot training loss and mAP progressions
res_img_p = run_folder / "results.png"
if res_img_p.exists():
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(cv2.imread(str(res_img_p)), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("YOLOv8 Training Loss & Metric Curves", fontsize=14)
    plt.show()

# 2. Plot detection confusion matrix
cm_p = run_folder / "confusion_matrix.png"
if cm_p.exists():
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(cv2.imread(str(cm_p)), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Object Detection Confusion Matrix", fontsize=14)
    plt.show()

# 3. Save structured training summary JSON
summary_data = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_architecture": "yolov8s",
    "class_name": "pneumonia_opacity",
    "detection_metrics_val": {
        "precision": p_val,
        "recall": r_val,
        "map50": map50_val,
        "map95": map95_val
    },
    "detection_metrics_test": {
        "precision": p_test,
        "recall": r_test,
        "map50": map50_test,
        "map95": map95_test
    },
    "classification_metrics_test": {
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "confusion_matrix": {"tp": tp, "fp": fp, "tn": tn, "fn": fn}
    },
    "weights": {
        "best": str(best_weights.resolve()) if best_weights.exists() else None,
        "last": str(last_checkpoint.resolve()) if last_checkpoint.exists() else None
    }
}

summary_json_path = run_folder / "training_summary.json"
with open(summary_json_path, "w") as f:
    json.dump(summary_data, f, indent=2)
print(f"Saved structured summary report: {summary_json_path}")

### 11. Download Model Weights (`best.pt`) for Apple M1 Deployment

Your trained model weights are saved at:
```text
/kaggle/working/rsna_pneumonia/yolov8s_pneumonia_run/weights/best.pt
```

**To transfer to your local machine:**
1. In the Kaggle Notebook right sidebar, click the **Output** tab.
2. Locate `rsna_pneumonia/yolov8s_pneumonia_run/weights/best.pt`.
3. Click **Download**.
4. Place the downloaded file into your local project repository at:
   ```text
   Disease_Detection_Iva/models/weights/best.pt
   ```
5. You can now execute local Apple Silicon M1 inference using Metal Performance Shaders (`mps`)!